# Prepare the pinned Qwen3-8B-Base model input

Run this once in a **CPU notebook with Internet on**. It downloads the
public model at the project's exact revision, writes its license and
snapshot manifest, and saves it under `/kaggle/working/qwen3-8b-base`.
No GPU is needed, and the project does not call a paid API.

After a successful Save & Run All, use Kaggle's Output tab to create a
**private dataset from the model folder**. Attach that dataset to the
separate frozen-feature notebook, which runs with **Internet off**.
Preserve `snapshot_manifest.json` beside `config.json` and all weights.
Review the upstream Apache-2.0 model license supplied with the snapshot.


In [ ]:
# Kaggle normally includes huggingface_hub. Install only if absent;
# this preparation notebook is the one workflow that uses the network.
import importlib.util
import subprocess
import sys
if importlib.util.find_spec("huggingface_hub") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "huggingface_hub"], check=True)
from importlib.metadata import version
print("huggingface_hub", version("huggingface_hub"))


In [ ]:
import hashlib
from pathlib import Path
import runpy

PREPARATION_SOURCE = '#!/usr/bin/env python3\n"""Prepare a pinned, offline model snapshot; --verify never accesses the network."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport re\nimport sys\nimport tempfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path, PurePosixPath\nfrom typing import Any, Callable\n\nDEFAULT_REPO = "Qwen/Qwen3-8B-Base"\nMANIFEST_NAME = "snapshot_manifest.json"\nMODEL_SPECS = {\n    DEFAULT_REPO: {\n        "revision": "49e3418fbbbca6ecbdf9608b4d22e5a407081db4",\n        "model_type": "qwen3", "hidden_size": 4096, "num_hidden_layers": 36,\n    },\n    "Qwen/Qwen3-14B-Base": {\n        "revision": "0b0bd3732e2c374d483664439ea334928b65f304",\n        "model_type": "qwen3", "hidden_size": 5120, "num_hidden_layers": 40,\n    },\n    "Qwen/Qwen3-4B-Base": {\n        "revision": "906bfd4b4dc7f14ee4320094d8b41684abff8539",\n        "model_type": "qwen3", "hidden_size": 2560, "num_hidden_layers": 36,\n    },\n    "microsoft/deberta-v3-large": {\n        "revision": "64a8c8eab3e352a784c658aef62be1662607476f",\n        "model_type": "deberta-v2", "hidden_size": 1024, "num_hidden_layers": 24,\n    },\n    "microsoft/deberta-v3-base": {\n        "revision": "8ccc9b6f36199bec6961081d44eb72fb3f7353f3",\n        "model_type": "deberta-v2", "hidden_size": 768, "num_hidden_layers": 12,\n    },\n    "StyleDistance/styledistance": {\n        "revision": "b7df5f0b0480773c097ba3121d83ca32b71015ca",\n        "model_type": "roberta", "hidden_size": 768, "num_hidden_layers": 12,\n    },\n    # Style and authorship encoders for the Claude-GLM specialist (docs/claude-glm.md).\n    # LUAR-MUD\'s own config nests its RoBERTa backbone; its remote code is not downloaded.\n    "rrivera1849/LUAR-MUD": {\n        "revision": "f1db50251805ed69b43cf4f72ea2f0e231f36a1c",\n        "model_type": "LUAR", "hidden_size": 768, "num_hidden_layers": 6,\n        "nested_config": "backbone_config",\n    },\n    "rrivera1849/LUSR": {\n        "revision": "ab7852825ad1be40112bbfdaf1f9e2315a1727b7",\n        "model_type": "roberta", "hidden_size": 768, "num_hidden_layers": 12,\n    },\n    "Qwen/Qwen3-Embedding-0.6B": {\n        "revision": "97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3",\n        "model_type": "qwen3", "hidden_size": 1024, "num_hidden_layers": 28,\n    },\n    "Qwen/Qwen3.5-9B": {\n        "revision": "c202236235762e1c871ad0ccb60c8ee5ba337b9a",\n        "model_type": "qwen3_5", "hidden_size": 4096, "num_hidden_layers": 32,\n        "nested_text": True,\n    },\n}\nMETADATA_PATTERNS = [\n    "*.json", "tokenizer*", "vocab*", "merges*", "spm.model",\n    "README*", "LICENSE*", "license*", "NOTICE*", "*.jinja",\n    "added_tokens*", "special_tokens*",\n]\n\n\nclass SnapshotError(ValueError):\n    """A snapshot cannot safely be prepared or verified."""\n\n\ndef _spec(repo_id: str, revision: str | None) -> tuple[dict[str, Any], str]:\n    if repo_id not in MODEL_SPECS:\n        raise SnapshotError("Model is outside the reviewed, pinned allowlist.")\n    spec = MODEL_SPECS[repo_id]\n    revision = spec["revision"] if revision is None else revision\n    if not isinstance(revision, str) or not re.fullmatch(r"[0-9a-f]{40}", revision):\n        raise SnapshotError("Revision must be an immutable 40-character lowercase commit SHA.")\n    if revision != spec["revision"]:\n        raise SnapshotError("Revision does not match the reviewed pin for this model.")\n    return spec, revision\n\n\ndef _read_json(path: Path) -> dict[str, Any]:\n    try:\n        data = json.loads(path.read_text(encoding="utf-8"))\n    except (OSError, UnicodeError, json.JSONDecodeError) as exc:\n        raise SnapshotError(f"Cannot read valid JSON from {path.name}.") from exc\n    if not isinstance(data, dict):\n        raise SnapshotError(f"{path.name} must contain a JSON object.")\n    return data\n\n\ndef _safe_relative(value: Any) -> str:\n    if not isinstance(value, str) or not value or "\\\\" in value:\n        raise SnapshotError("Manifest contains an invalid file path.")\n    path = PurePosixPath(value)\n    if not path.parts or path.is_absolute() or ".." in path.parts or str(path) != value:\n        raise SnapshotError("Manifest paths must be normalized relative paths.")\n    if path.parts[0] == ".cache" or value == MANIFEST_NAME:\n        raise SnapshotError("Manifest cannot inventory its cache or itself.")\n    return value\n\n\ndef _inventory(directory: Path) -> list[dict[str, Any]]:\n    files = []\n    for path in directory.rglob("*"):\n        relative = path.relative_to(directory)\n        if relative.parts[0] == ".cache" or relative.as_posix() == MANIFEST_NAME:\n            continue\n        if path.is_symlink():\n            raise SnapshotError("Snapshot artifacts must be local files, not symbolic links.")\n        if path.is_file():\n            files.append({"path": relative.as_posix(), "size": path.stat().st_size})\n    return sorted(files, key=lambda item: item["path"])\n\n\ndef _has_weights(files: list[dict[str, Any]]) -> bool:\n    return any(\n        item["path"].endswith(".safetensors")\n        or PurePosixPath(item["path"]).name == "pytorch_model.bin"\n        for item in files\n    )\n\n\ndef _validate_config(directory: Path, spec: dict[str, Any]) -> None:\n    config = _read_json(directory / "config.json")\n    if config.get("model_type") != spec["model_type"]:\n        raise SnapshotError("Downloaded config has an unexpected model_type.")\n    nested = spec.get("nested_config") or ("text_config" if spec.get("nested_text") else None)\n    text_config = config.get(nested, {}) if nested else config\n    for field in ("hidden_size", "num_hidden_layers"):\n        if not isinstance(text_config, dict) or text_config.get(field) != spec[field]:\n            raise SnapshotError(f"Downloaded config has an unexpected {field}.")\n\n\ndef _validate_shards(directory: Path, files: list[dict[str, Any]]) -> None:\n    names = {item["path"] for item in files}\n    for item in files:\n        if not item["path"].endswith(".safetensors.index.json"):\n            continue\n        index = _read_json(directory / item["path"])\n        weight_map = index.get("weight_map")\n        if not isinstance(weight_map, dict) or not weight_map:\n            raise SnapshotError("Weight index is missing its tensor-to-shard map.")\n        parent = PurePosixPath(item["path"]).parent\n        required = {_safe_relative((parent / _safe_relative(v)).as_posix())\n                    for v in weight_map.values()}\n        if not required.issubset(names):\n            raise SnapshotError("Snapshot is missing weight shards referenced by its index.")\n\n\ndef verify_snapshot(\n    output_dir: str | Path, *, repo_id: str | None = None, revision: str | None = None\n) -> dict[str, Any]:\n    """Check identity, inventory, byte sizes and config without importing Hub code."""\n    directory = Path(output_dir).expanduser().absolute()\n    if directory.is_symlink() or not directory.is_dir():\n        raise SnapshotError("Snapshot destination must be an existing local directory.")\n    manifest_path = directory / MANIFEST_NAME\n    if manifest_path.is_symlink():\n        raise SnapshotError("Snapshot manifest must be a local file.")\n    manifest = _read_json(manifest_path)\n    if type(manifest.get("schema_version")) is not int or manifest["schema_version"] != 1:\n        raise SnapshotError("Unsupported snapshot manifest schema_version.")\n    actual_repo = manifest.get("repo_id")\n    if not isinstance(actual_repo, str):\n        raise SnapshotError("Snapshot manifest is missing repo_id.")\n    if repo_id is not None and actual_repo != repo_id:\n        raise SnapshotError("Destination contains a different model; use a separate directory.")\n    spec, actual_revision = _spec(actual_repo, manifest.get("revision"))\n    if manifest.get("revision") != actual_revision:\n        raise SnapshotError("Snapshot manifest is missing its immutable revision.")\n    if revision is not None and actual_revision != revision:\n        raise SnapshotError("Destination contains a different revision; use a separate directory.")\n    if type(manifest.get("includes_weights")) is not bool:\n        raise SnapshotError("Manifest includes_weights must be a boolean.")\n    if not isinstance(manifest.get("downloaded_at_utc"), str):\n        raise SnapshotError("Manifest is missing downloaded_at_utc.")\n    entries = manifest.get("files")\n    if not isinstance(entries, list) or not entries:\n        raise SnapshotError("Manifest must contain a nonempty file inventory.")\n    declared = {}\n    for item in entries:\n        if not isinstance(item, dict):\n            raise SnapshotError("Invalid manifest inventory entry.")\n        name = _safe_relative(item.get("path"))\n        size = item.get("size")\n        if name in declared or type(size) is not int or size < 0:\n            raise SnapshotError("Manifest file paths must be unique and byte sizes nonnegative.")\n        declared[name] = size\n    actual_files = _inventory(directory)\n    actual = {item["path"]: item["size"] for item in actual_files}\n    if actual.keys() != declared.keys():\n        raise SnapshotError("Snapshot file count or paths differ from its manifest.")\n    if actual != declared:\n        raise SnapshotError("Snapshot file byte sizes differ from its manifest.")\n    if _has_weights(actual_files) != manifest["includes_weights"]:\n        raise SnapshotError("Snapshot weight files disagree with includes_weights.")\n    _validate_config(directory, spec)\n    if manifest["includes_weights"]:\n        _validate_shards(directory, actual_files)\n    return manifest\n\n\ndef _download_snapshot(**kwargs: Any) -> Any:\n    try:\n        from huggingface_hub import snapshot_download\n    except ImportError:\n        raise SnapshotError(\n            "Install huggingface_hub to download a snapshot; --verify needs only Python."\n        ) from None\n    return snapshot_download(**kwargs)\n\n\ndef _atomic_manifest(directory: Path, manifest: dict[str, Any]) -> None:\n    temporary = None\n    try:\n        with tempfile.NamedTemporaryFile(\n            mode="w", encoding="utf-8", dir=directory,\n            prefix=".snapshot_manifest.", suffix=".tmp", delete=False,\n        ) as handle:\n            temporary = Path(handle.name)\n            json.dump(manifest, handle, indent=2, sort_keys=True)\n            handle.write("\\n")\n            handle.flush()\n            os.fsync(handle.fileno())\n        os.replace(temporary, directory / MANIFEST_NAME)\n    finally:\n        if temporary is not None and temporary.exists():\n            temporary.unlink()\n\n\ndef prepare_snapshot(\n    output_dir: str | Path, *, repo_id: str = DEFAULT_REPO,\n    revision: str | None = None, metadata_only: bool = False,\n    downloader: Callable[..., Any] | None = None,\n) -> dict[str, Any]:\n    """Download reviewed artifacts; publish a manifest only after validation succeeds."""\n    spec, revision = _spec(repo_id, revision)\n    directory = Path(output_dir).expanduser().absolute()\n    if directory.is_symlink() or (directory.exists() and not directory.is_dir()):\n        raise SnapshotError("Snapshot destination must be a local directory.")\n    if directory.exists() and any(directory.iterdir()):\n        if not (directory / MANIFEST_NAME).is_file():\n            raise SnapshotError(\n                "Nonempty destination has no matching manifest; use a separate empty directory."\n            )\n        verify_snapshot(directory, repo_id=repo_id, revision=revision)\n    directory.mkdir(parents=True, exist_ok=True)\n    patterns = list(METADATA_PATTERNS)\n    if not metadata_only:\n        patterns.append("pytorch_model.bin" if repo_id.startswith("microsoft/")\n                        else "*.safetensors")\n    try:\n        (downloader or _download_snapshot)(\n            repo_id=repo_id, revision=revision, local_dir=str(directory),\n            allow_patterns=patterns, token=False, max_workers=4,\n        )\n    except SnapshotError:\n        raise\n    except Exception:\n        # Hub exception text can include authenticated URLs; never echo it.\n        raise SnapshotError(\n            "Snapshot download failed. Check network access and Hub availability. "\n            "No new manifest was published; use a fresh directory after a partial download."\n        ) from None\n    _validate_config(directory, spec)\n    files = _inventory(directory)\n    includes_weights = _has_weights(files)\n    if not metadata_only and not includes_weights:\n        raise SnapshotError("Download completed without model weights; no manifest was published.")\n    if includes_weights:\n        _validate_shards(directory, files)\n    manifest = {\n        "schema_version": 1, "repo_id": repo_id, "revision": revision,\n        "includes_weights": includes_weights, "files": files,\n        "downloaded_at_utc": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),\n    }\n    _atomic_manifest(directory, manifest)\n    return manifest\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--output-dir", required=True, type=Path)\n    parser.add_argument("--repo-id", choices=sorted(MODEL_SPECS))\n    parser.add_argument("--revision", help="Reviewed immutable commit SHA; defaults to the model\'s pin")\n    action = parser.add_mutually_exclusive_group()\n    action.add_argument("--metadata-only", action="store_true", help="Exclude model weight downloads")\n    action.add_argument("--verify", action="store_true", help="Verify an existing snapshot entirely offline")\n    args = parser.parse_args(argv)\n    try:\n        if args.verify:\n            manifest = verify_snapshot(args.output_dir, repo_id=args.repo_id, revision=args.revision)\n        else:\n            manifest = prepare_snapshot(\n                args.output_dir, repo_id=args.repo_id or DEFAULT_REPO,\n                revision=args.revision, metadata_only=args.metadata_only,\n            )\n    except (SnapshotError, OSError) as exc:\n        print(f"Model snapshot error: {exc}", file=sys.stderr)\n        return 1\n    print(json.dumps({key: manifest[key] for key in\n                      ("repo_id", "revision", "includes_weights")}\n                     | {"file_count": len(manifest["files"]),\n                        "verified_offline": args.verify}, indent=2))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
assert hashlib.sha256(PREPARATION_SOURCE.encode()).hexdigest() == '11d536381191a05224d21ff9b1ebd2757f18b6bb1cff6890095b5d9e619adee1'
helper_path = Path("/kaggle/working/prepare_model.py")
helper_path.write_text(PREPARATION_SOURCE, encoding="utf-8")
helper_namespace = runpy.run_path(str(helper_path), run_name="ntl_prepare_model")
preparation_result = helper_namespace["main"](["--output-dir", "/kaggle/working/qwen3-8b-base"])
if preparation_result not in (None, 0):
    raise RuntimeError(f"Model preparation failed with exit code {preparation_result}")


In [ ]:
import json
snapshot = Path("/kaggle/working/qwen3-8b-base")
manifest = json.loads((snapshot / "snapshot_manifest.json").read_text())
print("Repository:", manifest["repo_id"])
print("Revision:", manifest["revision"])
print("Files:", len(manifest["files"]))
print("Includes weights:", manifest["includes_weights"])
assert manifest["includes_weights"] is True
print("Create a private Kaggle dataset from:", snapshot)
